# TheLook 쇼핑몰 그래프 데이터셋 비즈니스 용어집 연동 및 관리

이 노트북은 구축된 정제 그래프 데이터셋(`thelook_network`)의 자산들과 비즈니스 용어집 간의 연동을 독립적으로 처리합니다. 

그래프 노드 및 엣지 테이블을 비즈니스 언어로 재정의하고, 대화형 분석 에이전트가 자연어로 그래프 탐색을 올바르게 수행할 수 있도록 용어 생성, 그래프 매핑 aspect 설정, 그리고 자산 연동 과정을 각 코드 단계별로 실습합니다.

### 학습 목표
1. **속성 그래프 비즈니스 자산화**: 물리적으로 흩어진 그래프 노드/엣지 테이블들을 표준화된 비즈니스 개념 용어로 용어집(Glossary)에 연결하는 아키텍처를 파악합니다.
2. **GQL 매핑 메타데이터 구현**: 대화형 에이전트가 자연어를 그래프 경로 GQL로 원활하게 번역할 수 있도록 도움을 주는 커스텀 Aspect Type(`graph-mapping`) 설계 및 매핑 주입 기법을 배웁니다.
3. **가상 리소스 자산 등록 (Entry Group & Entry)**: BigQuery가 자동 수집하지 못하는 속성 그래프 메타데이터 자산을 Knowledge Catalog에 직접 생성하고 연결하여 거버넌스 체계를 구현합니다.

### 작업 파이프라인

1. **환경 초기화**: GCP 프로젝트 자격 증명 획득 및 REST 호출을 위한 OAuth2 토큰을 발급받습니다.
2. **API 호출 공통 함수 정의**: 카테고리/용어 생성, 관계 연결, Aspect 업데이트 등 Knowledge Catalog 연동용 공통 헬퍼 함수를 정의합니다.
3. **용어집 및 Aspect Type 생성**: 그래프 전용 비즈니스 용어집과 GQL 매핑용 커스텀 Aspect Type(`graph-mapping`)을 확인 및 신규 생성합니다.
4. **그래프 용어 정의 데이터셋 로드**: GCS 버킷에 업로드된 CSV/JSON 파일로부터 그래프 비즈니스 정의와 GQL 매핑 메타데이터를 파싱합니다.
5. **그래프 비즈니스 용어 생성**: 정의된 구조에 따라 용어집 내에 카테고리와 용어 리소스 구조를 생성합니다.
6. **그래프 테이블 자산 및 비즈니스 용어 간 연동 수립**: 생성된 그래프 비즈니스 용어를 관련 BigQuery 테이블 및 컬럼과 논리적으로 매핑 연동합니다.
7. **그래프 자산 및 비즈니스 용어 간 직접 연동 수립 (Entry 기반)**: 가상 리소스 자산(Entry)을 생성하고 비즈니스 용어와 연동합니다.
8. **그래프 비즈니스 용어 Aspect 설정**: 각 용어에 개요(Overview) 및 GQL 매핑 Aspect를 부여합니다.

## Step 1: 초기 환경 설정 및 REST API 설정

In [ ]:
import json
import ssl
import urllib.request
import urllib.error
import time
import subprocess
import google.auth
from google.auth.transport.requests import Request

# 1. 설정 정보 정의
GLOSSARY_ID = "thelook-glossary"
LOCATION = "asia-northeast3"

# GCP 프로젝트 ID 조회 및 Access Token 발급
credentials, PROJECT_ID = google.auth.default()
credentials.refresh(Request())
ACCESS_TOKEN = credentials.token

ssl_context = ssl._create_unverified_context()

# REST API를 사용하여 프로젝트 번호(Project Number) 조회
req = urllib.request.Request(
    f"https://cloudresourcemanager.googleapis.com/v1/projects/{PROJECT_ID}",
    headers={"Authorization": f"Bearer {ACCESS_TOKEN}"}
)
with urllib.request.urlopen(req, context=ssl_context) as response:
    project_info = json.loads(response.read())
    PROJECT_NUMBER = project_info["projectNumber"]

print(f"GCP Project ID: {PROJECT_ID} (Project Number: {PROJECT_NUMBER})")

## Step 2: API 호출 공통 함수 정의

In [ ]:
existing_categories = None

# ==========================================
# [공통] 지수 백오프 및 재시도 내장 REST 요청 헬퍼
# ==========================================
def send_rest_request_with_retry(url, method="POST", body=None, max_attempts=5, ignore_409=True, retry_on_403_404=False):
    """
    백오프 및 특정 HTTP 에러 대응 로직을 중앙 집중화한 REST API 요청 공통 함수입니다.
    """
    headers = {
        "Authorization": f"Bearer {ACCESS_TOKEN}",
        "Content-Type": "application/json"
    }
    data = json.dumps(body).encode("utf-8") if body else None
    backoff = 2
    
    for attempt in range(max_attempts):
        req = urllib.request.Request(url, data=data, headers=headers, method=method)
        try:
            with urllib.request.urlopen(req, context=ssl_context) as res:
                return True
        except urllib.error.HTTPError as e:
            # 409 Conflict: 이미 존재하는 리소스인 경우 성공으로 간주
            if e.code == 409 and ignore_409:
                return True
            # 429 Rate Limit: 지수 백오프 대기 후 재시도
            elif e.code == 429:
                print(f"    [WAIT] 호출 제한(429)으로 인해 대기 후 재시도합니다. (시도 {attempt+1}/{max_attempts})...")
                time.sleep(backoff)
                backoff *= 2
            # 403/404: 인덱싱 미반영 대기 (Aspect 주입 단계 등에서 활성화 가능)
            elif e.code in (403, 404) and retry_on_403_404:
                print(f"    [WAIT] 인덱싱 반영 대기 중... 5초 후 재시도합니다. (시도 {attempt+1}/{max_attempts})")
                time.sleep(5)
            else:
                print(f"    [FAIL] API 호출 실패: {e.code} - {e.read().decode('utf-8')}")
                return False
                
    print(f"    [FAIL] API 호출 실패 (최대 시도 횟수 초과)")
    return False


# ==========================================
# 개별 리소스 관리 및 연동 함수
# ==========================================

# 1. 카테고리 생성 (gcloud CLI 기반)
def get_or_create_category(cat_id, display_name, parent_path):
    global existing_categories
    if existing_categories is None:
        existing_categories = set()
        list_cats = subprocess.run([
            "gcloud", "dataplex", "glossaries", "categories", "list",
            f"--glossary={GLOSSARY_ID}", f"--location={LOCATION}", f"--project={PROJECT_ID}",
            "--format=value(name)"
        ], capture_output=True, text=True)
        if list_cats.returncode == 0 and list_cats.stdout.strip():
            for cat_path in list_cats.stdout.strip().split("\n"):
                existing_categories.add(cat_path.split("/")[-1])
        print(f"기존 등록 카테고리 캐싱 완료 (기존 카테고리: {len(existing_categories)}개)")

    if cat_id not in existing_categories:
        print(f"카테고리 생성 중: {cat_id} ({display_name})...")
        subprocess.run([
            "gcloud", "dataplex", "glossaries", "categories", "create", cat_id,
            f"--glossary={GLOSSARY_ID}", f"--location={LOCATION}", f"--project={PROJECT_ID}",
            f"--parent={parent_path}",
            f"--display-name={display_name}"
        ], check=True)
        existing_categories.add(cat_id)
        print(f"  카테고리 생성 성공: {cat_id}. 인덱싱 대기 (3초)...")
        time.sleep(3)
    return f"projects/{PROJECT_NUMBER}/locations/{LOCATION}/glossaries/{GLOSSARY_ID}/categories/{cat_id}"

# 2. 용어 생성
def create_glossary_term(term_id, display_name, description, parent_path):
    url = f"https://dataplex.googleapis.com/v1/projects/{PROJECT_NUMBER}/locations/{LOCATION}/glossaries/{GLOSSARY_ID}/terms?termId={term_id}"
    body = {"displayName": display_name, "description": description, "parent": parent_path}
    return send_rest_request_with_retry(url, method="POST", body=body)

# 3. 테이블 연동
def link_term_to_table(term_entry_path, bq_entry_path, link_id):
    url = f"https://dataplex.googleapis.com/v1/projects/{PROJECT_NUMBER}/locations/{LOCATION}/entryGroups/@bigquery/entryLinks?entry_link_id={link_id}"
    body = {
        "entryLinkType": "projects/dataplex-types/locations/global/entryLinkTypes/definition",
        "entryReferences": [
            {"name": bq_entry_path, "type": "SOURCE"},
            {"name": term_entry_path, "type": "TARGET"}
        ]
    }
    return send_rest_request_with_retry(url, method="POST", body=body)

# 4. 연관 용어 상호 연동
def link_related_terms(term_1_path, term_2_path, link_id):
    url = f"https://dataplex.googleapis.com/v1/projects/{PROJECT_NUMBER}/locations/{LOCATION}/entryGroups/@dataplex/entryLinks?entry_link_id={link_id}"
    body = {
        "entryLinkType": "projects/dataplex-types/locations/global/entryLinkTypes/related",
        "entryReferences": [
            {"name": term_1_path, "type": "UNSPECIFIED"},
            {"name": term_2_path, "type": "UNSPECIFIED"}
        ]
    }
    return send_rest_request_with_retry(url, method="POST", body=body)

# 5. 동의어 상호 연동
def link_synonym_terms(term_1_path, term_2_path, link_id):
    url = f"https://dataplex.googleapis.com/v1/projects/{PROJECT_NUMBER}/locations/{LOCATION}/entryGroups/@dataplex/entryLinks?entry_link_id={link_id}"
    body = {
        "entryLinkType": "projects/dataplex-types/locations/global/entryLinkTypes/synonym",
        "entryReferences": [
            {"name": term_1_path, "type": "UNSPECIFIED"},
            {"name": term_2_path, "type": "UNSPECIFIED"}
        ]
    }
    return send_rest_request_with_retry(url, method="POST", body=body)

# 6. 개요 Aspect 등록
def set_term_overview(term_id, overview_text):
    term_entry_name = f"projects/{PROJECT_NUMBER}/locations/{LOCATION}/entryGroups/@dataplex/entries/projects/{PROJECT_NUMBER}/locations/{LOCATION}/glossaries/{GLOSSARY_ID}/terms/{term_id}"
    url = f"https://dataplex.googleapis.com/v1/{term_entry_name}?updateMask=aspects"
    body = {
        "aspects": {
            "dataplex-types.global.overview": {
                "aspectType": "projects/dataplex-types/locations/global/aspectTypes/overview",
                "data": {"content": overview_text, "links": [], "contentType": "MARKDOWN"}
            }
        }
    }
    return send_rest_request_with_retry(url, method="PATCH", body=body, max_attempts=10, retry_on_403_404=True)

# 7. 그래프 매핑 Aspect 등록
def set_term_graph_mapping(term_id, mapping_data):
    term_entry_name = f"projects/{PROJECT_NUMBER}/locations/{LOCATION}/entryGroups/@dataplex/entries/projects/{PROJECT_NUMBER}/locations/{LOCATION}/glossaries/{GLOSSARY_ID}/terms/{term_id}"
    url = f"https://dataplex.googleapis.com/v1/{term_entry_name}?updateMask=aspects"
    body = {
        "aspects": {
            f"{PROJECT_ID}.{LOCATION}.graph-mapping": {
                "aspectType": f"projects/{PROJECT_ID}/locations/{LOCATION}/aspectTypes/graph-mapping",
                "data": mapping_data
            }
        }
    }
    return send_rest_request_with_retry(url, method="PATCH", body=body, max_attempts=10, retry_on_403_404=True)

## Step 3: 용어집 및 Aspect Type 생성

In [ ]:
# Custom Aspect Type 및 Glossary 기반 확인 및 생성을 진행합니다.
def ensure_graph_mapping_aspect_type():
    describe_aspect = subprocess.run([
        "gcloud", "dataplex", "aspect-types", "describe", "graph-mapping",
        f"--location={LOCATION}", f"--project={PROJECT_ID}"
    ], capture_output=True, text=True)
    
    if describe_aspect.returncode != 0:
        print("\nCustom Aspect Type 'graph-mapping'이 존재하지 않습니다. 생성을 진행합니다...")
        import os
        from google.cloud import storage
        
        schema_path = "../resources/aspect_graph_mapping.json"
        if not os.path.exists(schema_path):
            schema_path = "analytics/resources/aspect_graph_mapping.json"
            if not os.path.exists(schema_path):
                schema_path = "resources/aspect_graph_mapping.json"
                
        if not os.path.exists(schema_path):
            print(f"로컬에서 스키마 파일을 찾지 못했습니다. GCS 버킷(gs://metadata-resources-{PROJECT_ID}/resources/aspect_graph_mapping.json)에서 다운로드합니다...")
            storage_client = storage.Client(project=PROJECT_ID)
            bucket = storage_client.bucket(f"metadata-resources-{PROJECT_ID}")
            blob = bucket.blob("resources/aspect_graph_mapping.json")
            if blob.exists():
                schema_path = "aspect_graph_mapping.json"
                blob.download_to_filename(schema_path)
                print(f"  GCS로부터 스키마 다운로드 완료: {schema_path}")
            else:
                raise FileNotFoundError("로컬 및 GCS 모두에서 aspect_graph_mapping.json 파일을 찾을 수 없습니다. 인프라 배포(Terraform) 상태를 확인하세요.")
                
        print(f"스키마 파일 로드 경로: {schema_path}")
        subprocess.run([
            "gcloud", "dataplex", "aspect-types", "create", "graph-mapping",
            f"--location={LOCATION}", f"--project={PROJECT_ID}",
            f"--metadata-template-file-name={schema_path}",
            "--display-name=Graph Mapping Ruleset",
            "--description=Ruleset for BigQuery Agent Text-to-GQL mapping."
        ], check=True)
        print("Custom Aspect Type 'graph-mapping' 생성 완료. 인덱싱을 위해 5초간 대기합니다...")
        time.sleep(5)
    else:
        print("Custom Aspect Type 'graph-mapping'이 이미 존재합니다.")

def check_glossary_exists():
    describe_glossary = subprocess.run([
        "gcloud", "dataplex", "glossaries", "describe", GLOSSARY_ID,
        f"--location={LOCATION}", f"--project={PROJECT_ID}"
    ], capture_output=True, text=True)
    return describe_glossary.returncode == 0

ensure_graph_mapping_aspect_type()

# 용어집이 없을 경우 신규로 생성하고, 존재하면 확인만 하고 계속 진행합니다.
if not check_glossary_exists():
    print(f"\n기본 Glossary '{GLOSSARY_ID}'가 존재하지 않아 생성을 진행합니다...")
    subprocess.run([
        "gcloud", "dataplex", "glossaries", "create", GLOSSARY_ID,
        f"--location={LOCATION}", f"--project={PROJECT_ID}",
        "--description=TheLook Ecommerce Business Glossary with Taxonomy",
        "--display-name=TheLook Glossary"
    ], check=True)
    print("용어집 생성 완료. 카탈로그 인덱싱 동기화를 위해 15초간 대기합니다...")
    time.sleep(15)
else:
    print(f"Glossary '{GLOSSARY_ID}'가 정상 확인되었습니다. 그래프 연동을 진행합니다.")

## Step 4: 그래프 용어 정의 데이터셋 로드

In [ ]:
# GCS 버킷 및 파일 경로 정의
RESOURCE_BUCKET = f"metadata-resources-{PROJECT_ID}"
GCS_BLOB_PATH = "resources/business_glossary_graph.json"
LOCAL_GLOSSARY_PATH = "../resources/business_glossary_graph.json"

import os
import json
from google.cloud import storage

# 로컬 파일이 존재하는 경우 우선적으로 로드 (로컬 개발 및 수정 테스트 시 유용)
if os.path.exists(LOCAL_GLOSSARY_PATH):
    print(f"로컬 파일 발견. 로컬 경로에서 그래프 용어집 데이터를 로드합니다: {LOCAL_GLOSSARY_PATH}")
    with open(LOCAL_GLOSSARY_PATH, "r", encoding="utf-8") as f:
        glossary_data = json.load(f)

# 로컬 파일이 없는 경우 (예: Colab 원격 실습 환경) -> GCS에서 다운로드하여 로드
else:
    print(f"로컬 파일이 없습니다. GCS 버킷(gs://{RESOURCE_BUCKET}/{GCS_BLOB_PATH})에서 데이터를 다운로드합니다...")
    storage_client = storage.Client(project=PROJECT_ID)
    bucket = storage_client.bucket(RESOURCE_BUCKET)
    blob = bucket.blob(GCS_BLOB_PATH)
    
    if blob.exists():
        glossary_data = json.loads(blob.download_as_text())
        print("  GCS로부터 데이터 로드 완료.")
    else:
        raise FileNotFoundError(f"로컬 및 GCS(gs://{RESOURCE_BUCKET}/{GCS_BLOB_PATH}) 모두에서 그래프 비즈니스 용어집 파일을 찾을 수 없습니다. 인프라 배포(Terraform) 상태를 확인하세요.")

print(f"총 {len(glossary_data)}개의 그래프 관련 용어/매핑 정의가 준비되었습니다.")

## Step 5: 그래프 비즈니스 용어 생성

In [ ]:
# First Pass: 그래프 관련 신규 용어 및 카테고리 등록
glossary_parent = f"projects/{PROJECT_NUMBER}/locations/{LOCATION}/glossaries/{GLOSSARY_ID}"

existing_terms = set()
list_terms = subprocess.run([
    "gcloud", "dataplex", "glossaries", "terms", "list",
    f"--glossary={GLOSSARY_ID}", f"--location={LOCATION}", f"--project={PROJECT_ID}",
    "--format=value(name)"
], capture_output=True, text=True)

if list_terms.returncode == 0 and list_terms.stdout.strip():
    for term_path in list_terms.stdout.strip().split("\n"):
        term_id = term_path.split("/")[-1]
        existing_terms.add(term_id)
print(f"기존 등록 용어 조회 완료 (기존 용어: {len(existing_terms)}개)")

# ----------------------------------------------------
# Pass 1: 모든 그래프 관련 카테고리 및 용어 생성
# ----------------------------------------------------
print("\n--- [Pass 1] 모든 그래프 카테고리 및 용어(Terms) 생성 시작 ---")
new_terms_created = False

for entry in glossary_data:
    if "term" not in entry:
        continue
        
    cat_id = entry["category_id"]
    cat_name = entry["category"]
    cat_path = get_or_create_category(cat_id, cat_name, glossary_parent)
    
    sub_cat_id = entry["sub_category_id"]
    sub_cat_name = entry["sub_category"]
    sub_cat_path = get_or_create_category(sub_cat_id, sub_cat_name, cat_path)
    
    term_name = entry["term"]
    term_id = entry["term_id"]
    description = entry.get("description", "")
    synonyms = entry.get("synonyms", [])
    
    # 1. 메인 용어 생성
    if term_id not in existing_terms:
        print(f"그래프 용어 등록 중 (REST API): {term_id} ({term_name}) under {sub_cat_id}...")
        if create_glossary_term(term_id, term_name, description, sub_cat_path):
            existing_terms.add(term_id)
            new_terms_created = True
            
    # 2. 동의어 용어 생성
    for idx, syn_name in enumerate(synonyms):
        syn_term_id = f"syn-{term_id}-{idx+1}"
        if syn_term_id not in existing_terms:
            syn_desc = f"[{term_name}의 동의어] {description}"
            print(f"  └─ 동의어 등록 중 (REST API): {syn_term_id} ({syn_name})...")
            if create_glossary_term(syn_term_id, syn_name, syn_desc, sub_cat_path):
                existing_terms.add(syn_term_id)
                new_terms_created = True

# 새롭게 생성된 용어가 있는 경우에만 인덱싱 대기를 위해 일시 대기
if new_terms_created:
    print("\n신규 용어가 생성되었습니다. Knowledge Catalog 검색 인덱싱 반영을 위해 15초간 대기합니다...")
    time.sleep(15)
else:
    print("\n새로 생성된 용어가 없어 대기 없이 즉시 진행합니다.")

## Step 6: 그래프 테이블 자산 및 비즈니스 용어 간 연동 수립

In [ ]:
# 안정적인 연동 수립을 위해 진입 전 대기
print("\n용어 생성 완료. 관계 연동 수립 진입 전 인덱싱 완료를 위해 10초간 대기합니다...")
time.sleep(10)

print("\n--- [그래프 데이터 자산 및 용어 간 연동 수립 (EntryLinks)] ---")
processed_links = set()

for entry in glossary_data:
    term_id = entry["term_id"]
    term_name = entry.get("term", term_id) # term 이름이 없는 경우 term_id 사용
    term_entry_path = f"projects/{PROJECT_NUMBER}/locations/{LOCATION}/entryGroups/@dataplex/entries/projects/{PROJECT_NUMBER}/locations/{LOCATION}/glossaries/{GLOSSARY_ID}/terms/{term_id}"
    
    # A. 테이블 자산 연동 (type = definition)
    related_tables = entry.get("related_tables", [])
    for table in related_tables:
        dataset_name, table_name = table.split(".")
        bq_entry_path = f"projects/{PROJECT_NUMBER}/locations/{LOCATION}/entryGroups/@bigquery/entries/bigquery.googleapis.com/projects/{PROJECT_ID}/datasets/{dataset_name}/tables/{table_name}"
        
        base_link_id = f"lk-{term_id}-to-{dataset_name}-{table_name}".lower().replace("_", "-")
        link_id = base_link_id[:63].rstrip("-")
        
        print(f"  └─ 그래프 테이블 연동 중: {table} ...")
        if link_term_to_table(term_entry_path, bq_entry_path, link_id):
            print(f"     [SUCCESS] 연동 완료 ({link_id})")
            
        # 동의어들도 동일한 그래프 테이블과 연동 수립 (Related entries 에 노출)
        synonyms = entry.get("synonyms", [])
        for idx, syn_name in enumerate(synonyms):
            syn_term_id = f"syn-{term_id}-{idx+1}"
            syn_entry_path = f"projects/{PROJECT_NUMBER}/locations/{LOCATION}/entryGroups/@dataplex/entries/projects/{PROJECT_NUMBER}/locations/{LOCATION}/glossaries/{GLOSSARY_ID}/terms/{syn_term_id}"
            
            syn_link_id = f"lk-{syn_term_id}-to-{dataset_name}-{table_name}".lower().replace("_", "-")
            syn_link_id = syn_link_id[:63].rstrip("-")
            
            print(f"  └─ [동의어] 그래프 테이블 연동 중: {syn_name} ({syn_term_id}) <-> {table} ...")
            if link_term_to_table(syn_entry_path, bq_entry_path, syn_link_id):
                print(f"     [SUCCESS] 동의어 그래프 연동 완료 ({syn_link_id})")

    # B. 비즈니스 연관 용어 간 연동 (새롭게 생성한 용어용 연관성 정의)
    related_terms = entry.get("related_terms", [])
    for rel_id in related_terms:
        sorted_ids = sorted([term_id, rel_id])
        link_key = tuple(sorted_ids)
        if link_key in processed_links:
            continue
            
        rel_entry_path = f"projects/{PROJECT_NUMBER}/locations/{LOCATION}/entryGroups/@dataplex/entries/projects/{PROJECT_NUMBER}/locations/{LOCATION}/glossaries/{GLOSSARY_ID}/terms/{rel_id}"
        base_link_id = f"lk-rel-{sorted_ids[0]}-to-{sorted_ids[1]}".lower().replace("_", "-")
        link_id = base_link_id[:63].rstrip("-")
        
        print(f"  └─ 연관 용어 연결 중: {term_id} <-> {rel_id} ...")
        if link_related_terms(term_entry_path, rel_entry_path, link_id):
            print(f"     [SUCCESS] 연관 완료 ({link_id})")
        processed_links.add(link_key)

    # C. 동의어 간 연동
    synonyms = entry.get("synonyms", [])
    for idx, syn_name in enumerate(synonyms):
        syn_term_id = f"syn-{term_id}-{idx+1}"
        syn_entry_path = f"projects/{PROJECT_NUMBER}/locations/{LOCATION}/entryGroups/@dataplex/entries/projects/{PROJECT_NUMBER}/locations/{LOCATION}/glossaries/{GLOSSARY_ID}/terms/{syn_term_id}"
        
        sorted_ids = sorted([term_id, syn_term_id])
        link_key = tuple(sorted_ids)
        if link_key in processed_links:
            continue
            
        base_link_id = f"lk-syn-{sorted_ids[0]}-to-{sorted_ids[1]}".lower().replace("_", "-")
        link_id = base_link_id[:63].rstrip("-")
        
        print(f"  └─ 동의어 연결 중: {term_name} <-> {syn_name} ...")
        if link_synonym_terms(term_entry_path, syn_entry_path, link_id):
            print(f"     [SUCCESS] 동의어 연동 완료 ({link_id})")
        processed_links.add(link_key)

print("\n모든 그래프 데이터 자산 및 용어 간 연동 정보가 정상적으로 업로드되었습니다!")

## Step 7: 그래프 자산 및 비즈니스 용어 간 직접 연동 수립 (Knowledge Catalog Entry 기반)

In [ ]:
# 1. 커스텀 Entry Group 생성 (graph-metadata)
print("--- [1단계: Entry Group 확인 및 생성] ---")
group_describe = subprocess.run([
    "gcloud", "dataplex", "entry-groups", "describe", "graph-metadata",
    f"--location={LOCATION}", f"--project={PROJECT_ID}"
], capture_output=True, text=True)

if group_describe.returncode != 0:
    print("커스텀 Entry Group 'graph-metadata'이 존재하지 않습니다. 생성을 진행합니다...")
    subprocess.run([
        "gcloud", "dataplex", "entry-groups", "create", "graph-metadata",
        f"--location={LOCATION}", f"--project={PROJECT_ID}",
        "--description=Custom entry group for Graph Metadata"
    ], check=True)
    print("Entry Group 생성 완료.")
else:
    print("Entry Group 'graph-metadata'이 이미 존재합니다.")

# 2. 커스텀 Entry Type 생성 (bigquery-graph)
print("\n--- [2단계: Entry Type 확인 및 생성] ---")
type_describe = subprocess.run([
    "gcloud", "dataplex", "entry-types", "describe", "bigquery-graph",
    f"--location={LOCATION}", f"--project={PROJECT_ID}"
], capture_output=True, text=True)

if type_describe.returncode != 0:
    print("커스텀 Entry Type 'bigquery-graph'이 존재하지 않습니다. 생성을 진행합니다...")
    subprocess.run([
        "gcloud", "dataplex", "entry-types", "create", "bigquery-graph",
        f"--location={LOCATION}", f"--project={PROJECT_ID}",
        "--description=Entry type for BigQuery Property Graphs",
        "--display-name=BigQuery Property Graph"
    ], check=True)
    print("Entry Type 생성 완료.")
else:
    print("Entry Type 'bigquery-graph'이 이미 존재합니다.")

# 3. 속성 그래프 Entry 생성 (product_recommendation_graph)
print("\n--- [3단계: 속성 그래프 Entry 확인 및 생성] ---")
entry_describe = subprocess.run([
    "gcloud", "dataplex", "entries", "describe", "product_recommendation_graph",
    "--entry-group=graph-metadata", f"--location={LOCATION}", f"--project={PROJECT_ID}"
], capture_output=True, text=True)

if entry_describe.returncode != 0:
    print("속성 그래프 Entry 'product_recommendation_graph'이 존재하지 않습니다. 생성을 진행합니다...")
    # Entry 생성에 필요한 시간 생성
    current_time = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
    subprocess.run([
        "gcloud", "dataplex", "entries", "create", "product_recommendation_graph",
        "--entry-group=graph-metadata", f"--location={LOCATION}", f"--project={PROJECT_ID}",
        f"--entry-type=projects/{PROJECT_ID}/locations/{LOCATION}/entryTypes/bigquery-graph",
        f"--fully-qualified-name=bigquery:{PROJECT_ID}.thelook_network.product_recommendation_graph",
        f"--entry-source-resource=projects/{PROJECT_ID}/datasets/thelook_network/graphs/product_recommendation_graph",
        "--entry-source-system=BIGQUERY",
        "--entry-source-platform=GCP",
        "--entry-source-display-name=Product Recommendation Graph",
        "--entry-source-description=BigQuery Property Graph representing product recommendation relationships.",
        f"--entry-source-update-time={current_time}"
    ], check=True)
    print("속성 그래프 Entry 생성 완료.")
else:
    print("속성 그래프 Entry 'product_recommendation_graph'이 이미 존재합니다.")

# 4. 비즈니스 용어와 속성 그래프 간 직접 연동 수립 (EntryLinks)
print("\n--- [4단계: 비즈니스 용어 및 속성 그래프 간 연동 수립 (EntryLinks)] ---")

# SSL 검증 실패 우회를 위한 ssl context 생성 (필요한 경우 대비)
import ssl
ssl_context = ssl._create_unverified_context()

def link_term_to_graph(term_entry_path, graph_entry_path, link_id):
    url = f"https://dataplex.googleapis.com/v1/projects/{PROJECT_NUMBER}/locations/{LOCATION}/entryGroups/graph-metadata/entryLinks?entry_link_id={link_id}"
    headers = {
        "Authorization": f"Bearer {ACCESS_TOKEN}",
        "Content-Type": "application/json"
    }
    body = {
        "entryLinkType": "projects/dataplex-types/locations/global/entryLinkTypes/definition",
        "entryReferences": [
            {"name": graph_entry_path, "type": "SOURCE"},
            {"name": term_entry_path, "type": "TARGET"}
        ]
    }
    
    backoff = 2
    for attempt in range(5):
        req = urllib.request.Request(url, data=json.dumps(body).encode(), headers=headers, method="POST")
        try:
            with urllib.request.urlopen(req, context=ssl_context) as res:
                return True
        except urllib.error.HTTPError as e:
            if e.code == 409:
                return True
            elif e.code == 429:
                print(f"    [WAIT] 호출 제한(429)으로 인해 대기 후 재시도합니다. (시도 {attempt+1}/5)...")
                time.sleep(backoff)
                backoff *= 2
            else:
                print(f"    [FAIL] Graph Link {link_id} 생성 실패: {e.code} - {e.read().decode()}")
                return False
    print(f"    [FAIL] Graph Link {link_id} 생성 실패 (시간 초과)")
    return False

graph_entry_path = f"projects/{PROJECT_NUMBER}/locations/{LOCATION}/entryGroups/graph-metadata/entries/product_recommendation_graph"

for entry in glossary_data:
    if "term_id" not in entry:
        continue
    term_id = entry["term_id"]
    term_entry_path = f"projects/{PROJECT_NUMBER}/locations/{LOCATION}/entryGroups/@dataplex/entries/projects/{PROJECT_NUMBER}/locations/{LOCATION}/glossaries/{GLOSSARY_ID}/terms/{term_id}"
    
    link_id = f"lk-{term_id}-to-graph-prod-rec".lower().replace("_", "-")
    link_id = link_id[:63].rstrip("-")
    
    print(f"  └─ 그래프 자산 직접 연동 중: term '{term_id}' ...")
    if link_term_to_graph(term_entry_path, graph_entry_path, link_id):
        print(f"     [SUCCESS] 연동 완료 ({link_id})")
        
    # 동의어에 대한 연동도 수립
    synonyms = entry.get("synonyms", [])
    for idx, syn_name in enumerate(synonyms):
        syn_term_id = f"syn-{term_id}-{idx+1}"
        syn_entry_path = f"projects/{PROJECT_NUMBER}/locations/{LOCATION}/entryGroups/@dataplex/entries/projects/{PROJECT_NUMBER}/locations/{LOCATION}/glossaries/{GLOSSARY_ID}/terms/{syn_term_id}"
        syn_link_id = f"lk-{syn_term_id}-to-graph-prod-rec".lower().replace("_", "-")
        syn_link_id = syn_link_id[:63].rstrip("-")
        
        print(f"  └─ [동의어] 그래프 자산 직접 연동 중: {syn_name} ({syn_term_id}) ...")
        if link_term_to_graph(syn_entry_path, graph_entry_path, syn_link_id):
            print(f"     [SUCCESS] 동의어 그래프 연동 완료 ({syn_link_id})")

print("\n모든 그래프 자산 직접 연동 정보가 정상적으로 수립되었습니다!")

## Step 8: 그래프 비즈니스 용어 Aspect(메타데이터) 설정

In [ ]:
# ----------------------------------------------------
# Pass 2: 모든 그래프 용어에 대해 aspect 설정
# ----------------------------------------------------
for entry in glossary_data:
    if "term" not in entry:
        continue
        
    term_id = entry["term_id"]
    overview = entry.get("overview", "")
    graph_mapping = entry.get("graph_mapping", None)
    synonyms = entry.get("synonyms", [])
    
    # 1. 메인 용어 aspect 설정
    if overview:
        set_term_overview(term_id, overview)
            
    if graph_mapping:
        set_term_graph_mapping(term_id, graph_mapping)
            
    # 2. 동의어 aspect 설정 (메인 용어의 aspect 상속)
    for idx, syn_name in enumerate(synonyms):
        syn_term_id = f"syn-{term_id}-{idx+1}"
        
        if overview:
            set_term_overview(syn_term_id, overview)
                
        if graph_mapping:
            set_term_graph_mapping(syn_term_id, graph_mapping)

print("모든 그래프 비즈니스 용어에 대한 Aspect(개요 및 Graph 매핑) 설정이 완료되었습니다.")

## Appendix: 대화형 그래프 분석 검증 시나리오 및 GQL 그래프 매핑 가이드

등록된 Custom Aspect인 **`graph-mapping`**은 대화형 분석 에이전트가 GQL을 작성하거나 속성 그래프 구조를 해석할 때 오차가 없도록 그래프 패턴 스니펫을 대입해 주는 가이드 역할을 수행합니다.

### 1. 그래프 용어별 그래프 매핑 조견표

| 용어 ID | 용어명 | 매핑 종류 | 타겟 테이블 / 패턴 | 등록된 매핑 규칙 설명 |
| :--- | :--- | :--- | :--- | :--- |
| **`co-purchased-products`** | 연관 상품 | `Graph_Edge` | `thelook_network.edge_co_purchased` (label: `co_purchased_with`) | 동일 주문에서 동시에 2회 이상 결제된 상품 간의 그래프 엣지 관계 정의 |
| **`recommended-products`** | 추천 상품 | `Graph_Path` | `(u1:Customer)-[:ordered]->(p1:Product)<-[:ordered]-(u2:Customer)-[:ordered]->(p2:Product)` | 유저-상품 간의 다중 홉 관계(주문 공유)를 이용한 협업 필터링 추천 경로 명세 |

---

### 2. 에이전트 검증용 자연어 질문 및 예상 GQL 쿼리

#### Q1. [동시구매 연관 상품 분석] `"상품 ID 19666과 자주 같이 구매되는 연관 상품 목록을 보여줘."`
* **예상 결과 GQL:**
```sql
SELECT 
  product_name AS recommended_product,
  co_purchase_count
FROM GRAPH_TABLE(
  thelook_network.product_recommendation_graph
  MATCH (p1:Product)-[co:co_purchased_with]->(p2:Product)
  WHERE p1.product_id = 19666
  RETURN p2.product_name AS product_name, co.co_purchase_count AS co_purchase_count
)
ORDER BY co_purchase_count DESC
LIMIT 10;
```
* **🔍 주요 검증 포인트:**
  1. **용어 매핑:** 자연어의 **"연관 상품"**이 비즈니스 용어 **"연관 상품" (ID: `co-purchased-products`)**으로 파악되었는가?
  2. **그래프 엣지 번역:** `graph-mapping`에 명세된 `type: Graph_Edge`와 `label: co_purchased_with` 정보를 읽어 `MATCH (p1:Product)-[:co_purchased_with]->(p2:Product)` 패턴을 올바르게 도출하였는가?

#### Q2. [협업 필터링 추천] `"유저 ID 367번 고객에게 추천 상품 목록 10개를 뽑아줘."`
* **예상 결과 GQL:**
```sql
SELECT 
  recommended_product,
  COUNT(*) AS recommendation_strength
FROM GRAPH_TABLE(
  thelook_network.product_recommendation_graph
  MATCH (u1:Customer)-[:ordered]->(p1:Product)<-[:ordered]-(u2:Customer)-[:ordered]->(p2:Product)
  WHERE u1.user_id = 367 AND p1.product_id != p2.product_id
  RETURN p2.product_name AS recommended_product
)
GROUP BY recommended_product
ORDER BY recommendation_strength DESC
LIMIT 10;
```
* **🔍 주요 검증 포인트:**
  1. **용어 매핑:** 자연어의 **"추천 상품"**이 비즈니스 용어 **"추천 상품" (ID: `recommended-products`)**으로 매핑되었는가?
  2. **복합 그래프 경로 주입:** 에이전트가 `graph-mapping`에 저장된 `pattern` 값 `(u1:Customer)-[:ordered]->(p1:Product)<-[:ordered]-(u2:Customer)-[:ordered]->(p2:Product)`을 직접 조회하여 GQL의 `MATCH` 문에 그대로 삽입하고 유저 필터(`u1.user_id = 367`)를 바인딩했는가?

---

### 3. Knowledge Catalog 및 용어집 연동의 비즈니스적 가치
1. **GQL 번역 성능 극대화**: 복잡한 그래프 홉 쿼리를 작성할 때 경로 방향이나 라벨명을 잘못 유추하는 경우가 빈번합니다. Knowledge Catalog 용어집의 `graph-mapping`은 검증된 그래프 통로 패턴을 제공하여 GQL 생성 성공률을 비약적으로 높집니다.
2. **비즈니스 용어의 통일**: 데이터 분석가, AI 에이전트, 현업 임직원 모두가 "함께 산 상품", "동시 구매 제품", "Cross-sell" 등의 다른 용어를 사용하더라도 내부적으로는 `co-purchased-products` 라는 표준화된 자산으로 수렴되도록 보장합니다.
3. **데이터 거버넌스 시각화**: 특정 그래프 테이블이 수정되었을 때, 해당 테이블과 연관된 비즈니스 개념 및 AI 카탈로그 상의 영향도를 즉각 파악할 수 있는 계보 및 거버넌스 토대를 제공합니다.